In [ ]:
# =============================================================================
# CÉLULA 0 — METADATA (cole esta célula como Raw / Markdown no Colab)
# =============================================================================
# FOOTBALL TACTICAL AI — Proof of Concept
# =========================================
# Autor: Projeto educacional
# Stack: Python · OpenCV · YOLOv8 · NetworkX · Qwen (via Transformers)
# Ambiente: Google Colab (GPU T4 recomendada)
# Licença: MIT — uso livre para fins educacionais


In [ ]:
# =============================================================================
# CÉLULA 1 — SETUP & INSTALAÇÃO
# =============================================================================
# Instala todas as dependências necessárias.
# Execute esta célula apenas uma vez por sessão do Colab.

!pip install -q ultralytics opencv-python-headless matplotlib numpy networkx \
    Pillow scipy scikit-learn transformers accelerate torch torchvision \
    lap

# Para tracking ByteTrack (integrado na Ultralytics >= 8.0)
# Nenhuma instalação extra necessária.

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# =============================================================================
# CÉLULA 2 — IMPORTS & CONFIGURAÇÃO GLOBAL
# =============================================================================

import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch
from PIL import Image, ImageDraw, ImageFont
import networkx as nx
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import defaultdict
import json
import os
import warnings
import textwrap
from scipy.spatial import ConvexHull
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")

# ── Configuração Global ──────────────────────────────────────────────────────
@dataclass
class Config:
    """Parâmetros centrais do pipeline."""
    # Vídeo
    video_path: str = "football_clip.mp4"
    fps_sample: int = 2                       # frames por segundo a extrair
    max_duration: int = 39                     # duração máx. em segundos

    # YOLO
    yolo_model: str = "yolov8n.pt"            # nano para velocidade no Colab
    yolo_conf: float = 0.30                   # confiança mínima
    yolo_person_class: int = 0                # classe "person" no COCO
    yolo_ball_class: int = 32                 # classe "sports ball" no COCO

    # Campo
    field_length: float = 105.0               # metros
    field_width: float = 68.0                 # metros

    # Cores dos times (BGR para OpenCV, mas usaremos RGB no matplotlib)
    team_a_color: str = "#FFD700"             # Brasil (amarelo)
    team_b_color: str = "#1E90FF"             # Adversário (azul)
    ball_color: str = "#FF4444"               # Bola (vermelho)

    # LLM
    llm_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    llm_max_tokens: int = 1500
    llm_temperature: float = 0.7

    # Output
    output_dir: str = "output"

cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)
print("✅ Configuração carregada.")


In [ ]:
# =============================================================================
# CÉLULA 3 — EXTRAÇÃO DE FRAMES DO VÍDEO
# =============================================================================

def extract_frames(video_path: str, fps_sample: int, max_duration: int) -> List[Tuple[int, float, np.ndarray]]:
    """
    Extrai frames do vídeo a uma taxa fixa.
    Retorna lista de (frame_index, timestamp_s, frame_bgr).
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Não foi possível abrir: {video_path}")

    video_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / video_fps if video_fps > 0 else 0
    duration = min(duration, max_duration)

    frame_interval = max(1, int(video_fps / fps_sample))
    frames = []
    idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        timestamp = idx / video_fps
        if timestamp > max_duration:
            break
        if idx % frame_interval == 0:
            frames.append((idx, round(timestamp, 2), frame))
        idx += 1

    cap.release()
    print(f"📹 Vídeo: {duration:.1f}s | FPS original: {video_fps:.0f} | "
          f"Frames extraídos: {len(frames)} (a cada {frame_interval} frames)")
    return frames


# ── Upload / Demo ─────────────────────────────────────────────────────────────
# Se estiver no Colab, faça upload do vídeo ou use o snippet abaixo para
# baixar um vídeo demo (substitua a URL por uma válida).

# Para upload manual no Colab descomente:
from google.colab import files
uploaded = files.upload()
cfg.video_path = list(uploaded.keys())[0]

# Para gerar um vídeo sintético de demonstração (caso não tenha um real):
def generate_synthetic_video(path: str, duration: int = 10, fps: int = 30):
    """Gera um vídeo sintético com pontos simulando jogadores."""
    w, h = 960, 540
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(path, fourcc, fps, (w, h))
    rng = np.random.RandomState(42)

    # Posições iniciais: 11 time A + 11 time B + 1 bola
    team_a = rng.rand(11, 2) * [w * 0.45, h * 0.8] + [w * 0.05, h * 0.1]
    team_b = rng.rand(11, 2) * [w * 0.45, h * 0.8] + [w * 0.50, h * 0.1]
    ball = np.array([[w / 2, h / 2]], dtype=float)

    for f in range(duration * fps):
        frame = np.full((h, w, 3), (34, 120, 40), dtype=np.uint8)  # verde campo
        # Linhas do campo
        cv2.rectangle(frame, (30, 30), (w - 30, h - 30), (255, 255, 255), 2)
        cv2.line(frame, (w // 2, 30), (w // 2, h - 30), (255, 255, 255), 2)
        cv2.circle(frame, (w // 2, h // 2), 60, (255, 255, 255), 2)

        # Movimento suave
        team_a += rng.randn(11, 2) * 1.5
        team_b += rng.randn(11, 2) * 1.5
        ball += rng.randn(1, 2) * 3.0

        team_a = np.clip(team_a, [35, 35], [w - 35, h - 35])
        team_b = np.clip(team_b, [35, 35], [w - 35, h - 35])
        ball = np.clip(ball, [35, 35], [w - 35, h - 35])

        for p in team_a:
            cv2.circle(frame, tuple(p.astype(int)), 14, (0, 215, 255), -1)
            cv2.circle(frame, tuple(p.astype(int)), 14, (0, 0, 0), 2)
        for p in team_b:
            cv2.circle(frame, tuple(p.astype(int)), 14, (255, 144, 30), -1)
            cv2.circle(frame, tuple(p.astype(int)), 14, (0, 0, 0), 2)
        for p in ball:
            cv2.circle(frame, tuple(p.astype(int)), 8, (0, 0, 255), -1)

        writer.write(frame)
    writer.release()
    print(f"🎬 Vídeo sintético gerado: {path} ({duration}s, {fps} fps)")

# Gera vídeo sintético se o real não existir
if not os.path.exists(cfg.video_path):
    print("⚠️  Vídeo não encontrado. Gerando vídeo sintético de demonstração...")
    generate_synthetic_video(cfg.video_path, duration=min(15, cfg.max_duration), fps=30)

frames = extract_frames(cfg.video_path, cfg.fps_sample, cfg.max_duration)


In [ ]:
# =============================================================================
# CÉLULA 4 — DETECÇÃO DE JOGADORES E BOLA COM YOLO
# =============================================================================

from ultralytics import YOLO

@dataclass
class Detection:
    """Uma detecção individual."""
    bbox: Tuple[int, int, int, int]   # x1, y1, x2, y2
    center: Tuple[float, float]
    confidence: float
    class_id: int
    team: Optional[str] = None        # 'team_a', 'team_b', 'ball'
    track_id: Optional[int] = None

def detect_objects(frames: List[Tuple[int, float, np.ndarray]],
                   config: Config) -> Dict[float, List[Detection]]:
    """
    Executa YOLOv8 em cada frame para detectar jogadores e bola.
    Retorna dict: timestamp → lista de detecções.
    """
    model = YOLO(config.yolo_model)
    all_detections: Dict[float, List[Detection]] = {}

    for idx, timestamp, frame in frames:
        results = model(frame, conf=config.yolo_conf, verbose=False)
        dets = []
        for r in results:
            for box in r.boxes:
                cls_id = int(box.cls[0])
                if cls_id not in (config.yolo_person_class, config.yolo_ball_class):
                    continue
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                conf = float(box.conf[0])
                det = Detection(
                    bbox=(x1, y1, x2, y2),
                    center=(cx, cy),
                    confidence=conf,
                    class_id=cls_id,
                    team="ball" if cls_id == config.yolo_ball_class else None
                )
                dets.append(det)
        all_detections[timestamp] = dets

    total = sum(len(v) for v in all_detections.values())
    print(f"🔍 Detecção concluída: {total} objetos em {len(all_detections)} frames")
    return all_detections


detections_by_frame = detect_objects(frames, cfg)


In [ ]:
# =============================================================================
# CÉLULA 5 — CLASSIFICAÇÃO DE TIMES + TRACKING SIMPLIFICADO
# =============================================================================

def extract_jersey_color(frame: np.ndarray, bbox: Tuple[int, int, int, int]) -> np.ndarray:
    """Extrai a cor dominante da região do torso do jogador."""
    x1, y1, x2, y2 = bbox
    h = y2 - y1
    # Região do torso (terço superior-central do bbox)
    torso_y1 = y1 + int(h * 0.15)
    torso_y2 = y1 + int(h * 0.55)
    torso = frame[torso_y1:torso_y2, x1:x2]
    if torso.size == 0:
        return np.array([128, 128, 128])
    # Converter para HSV para melhor separação de cores
    hsv = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
    return np.median(hsv.reshape(-1, 3), axis=0)


def classify_teams(frames: List[Tuple[int, float, np.ndarray]],
                   detections: Dict[float, List[Detection]]) -> Dict[float, List[Detection]]:
    """
    Usa K-Means (k=2) nas cores das camisas para separar os jogadores
    em dois times. O time com camisa mais 'amarela' em HSV é o Team A (Brasil).
    """
    all_colors = []
    det_indices = []  # (timestamp, det_idx)

    frame_lookup = {ts: frm for _, ts, frm in frames}

    for ts, dets in detections.items():
        frame = frame_lookup.get(ts)
        if frame is None:
            continue
        for i, det in enumerate(dets):
            if det.team == "ball":
                continue
            color = extract_jersey_color(frame, det.bbox)
            all_colors.append(color)
            det_indices.append((ts, i))

    if len(all_colors) < 4:
        print("⚠️  Poucos jogadores detectados para classificação de times.")
        return detections

    colors_array = np.array(all_colors)
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10).fit(colors_array)
    labels = kmeans.labels_
    centers = kmeans.cluster_centers_

    # Cluster com Hue mais próximo de amarelo (H ≈ 20-35 em HSV OpenCV) = Team A
    hue_diff_a = min(abs(centers[0][0] - 25), abs(centers[0][0] - 25 + 180))
    hue_diff_b = min(abs(centers[1][0] - 25), abs(centers[1][0] - 25 + 180))
    team_a_label = 0 if hue_diff_a < hue_diff_b else 1

    for (ts, det_i), label in zip(det_indices, labels):
        detections[ts][det_i].team = "team_a" if label == team_a_label else "team_b"

    # Contagem
    counts = {"team_a": 0, "team_b": 0, "ball": 0}
    for dets in detections.values():
        for d in dets:
            if d.team in counts:
                counts[d.team] += 1
    print(f"🏃 Classificação: Team A (Brasil): {counts['team_a']} detecções | "
          f"Team B (Adversário): {counts['team_b']} | Bola: {counts['ball']}")
    return detections


# ── Tracking simplificado por proximidade (IOU-free) ──────────────────────
def simple_track(detections: Dict[float, List[Detection]],
                 max_dist: float = 80.0) -> Dict[float, List[Detection]]:
    """
    Tracking simplificado: associa detecções entre frames consecutivos
    por distância euclidiana mínima. Atribui track_id único.
    """
    timestamps = sorted(detections.keys())
    next_id = 0
    prev_dets: List[Detection] = []

    for ts in timestamps:
        curr_dets = detections[ts]
        if not prev_dets:
            for d in curr_dets:
                d.track_id = next_id
                next_id += 1
        else:
            used_prev = set()
            for d in curr_dets:
                best_dist = max_dist
                best_idx = -1
                for pi, pd in enumerate(prev_dets):
                    if pi in used_prev or pd.team != d.team:
                        continue
                    dist = np.hypot(d.center[0] - pd.center[0],
                                    d.center[1] - pd.center[1])
                    if dist < best_dist:
                        best_dist = dist
                        best_idx = pi
                if best_idx >= 0:
                    d.track_id = prev_dets[best_idx].track_id
                    used_prev.add(best_idx)
                else:
                    d.track_id = next_id
                    next_id += 1
        prev_dets = curr_dets

    print(f"🔗 Tracking concluído: {next_id} IDs únicos atribuídos")
    return detections


detections_by_frame = classify_teams(frames, detections_by_frame)
detections_by_frame = simple_track(detections_by_frame)


In [ ]:
# =============================================================================
# CÉLULA 6 — FIELD MAPPING (Projeção para campo 2D)
# =============================================================================

@dataclass
class FieldPosition:
    """Posição normalizada de um jogador no campo 2D."""
    x: float           # 0..105 (comprimento)
    y: float           # 0..68  (largura)
    team: str
    track_id: int
    timestamp: float

def map_to_field(detections: Dict[float, List[Detection]],
                 frames: List[Tuple[int, float, np.ndarray]],
                 config: Config) -> Dict[float, List[FieldPosition]]:
    """
    Projeção simplificada: mapeia coordenadas de pixel para o campo
    normalizado 105×68 usando escala linear.
    (Para produção, usar homografia com pontos de calibração.)
    """
    # Obter dimensões do primeiro frame
    _, _, sample_frame = frames[0]
    img_h, img_w = sample_frame.shape[:2]

    field_positions: Dict[float, List[FieldPosition]] = {}

    for ts, dets in detections.items():
        positions = []
        for d in dets:
            # Pé do jogador ≈ centro-baixo do bbox
            foot_x = d.center[0]
            foot_y = d.bbox[3]  # y2 = base do bbox

            # Escala linear (simplificação)
            fx = (foot_x / img_w) * config.field_length
            fy = (foot_y / img_h) * config.field_width

            fx = np.clip(fx, 0, config.field_length)
            fy = np.clip(fy, 0, config.field_width)

            positions.append(FieldPosition(
                x=round(fx, 1),
                y=round(fy, 1),
                team=d.team or "unknown",
                track_id=d.track_id or 0,
                timestamp=ts
            ))
        field_positions[ts] = positions

    print(f"⚽ Field Mapping concluído: {len(field_positions)} frames mapeados")
    return field_positions


field_data = map_to_field(detections_by_frame, frames, cfg)


In [ ]:
# =============================================================================
# CÉLULA 7 — GRAFO TÁTICO E MÉTRICAS
# =============================================================================

@dataclass
class TacticalMetrics:
    """Métricas táticas calculadas para um frame."""
    timestamp: float
    team_a_centroid: Tuple[float, float] = (0, 0)
    team_b_centroid: Tuple[float, float] = (0, 0)
    team_a_compactness: float = 0.0
    team_b_compactness: float = 0.0
    team_a_width: float = 0.0
    team_b_width: float = 0.0
    team_a_depth: float = 0.0
    team_b_depth: float = 0.0
    free_zones: List[Tuple[float, float]] = field(default_factory=list)
    numerical_superiority: Dict[str, str] = field(default_factory=dict)
    passing_lanes_blocked: int = 0
    graph_edges: List[Tuple[int, int, str]] = field(default_factory=list)


def build_tactical_graph(positions: List[FieldPosition],
                         proximity_thresh: float = 15.0) -> Tuple[nx.Graph, TacticalMetrics]:
    """
    Constrói grafo tático e calcula métricas para um frame.

    Nós: jogadores (com atributos de time e posição)
    Arestas:
      - 'teammate': entre jogadores do mesmo time próximos (linha de passe)
      - 'pressure': entre adversários próximos (marcação)
    """
    G = nx.Graph()
    metrics = TacticalMetrics(timestamp=positions[0].timestamp if positions else 0)

    team_a_pos = []
    team_b_pos = []

    # Adicionar nós
    for p in positions:
        G.add_node(p.track_id, x=p.x, y=p.y, team=p.team)
        if p.team == "team_a":
            team_a_pos.append((p.x, p.y))
        elif p.team == "team_b":
            team_b_pos.append((p.x, p.y))

    # Adicionar arestas
    edges = []
    for i, p1 in enumerate(positions):
        for j, p2 in enumerate(positions):
            if j <= i:
                continue
            dist = np.hypot(p1.x - p2.x, p1.y - p2.y)
            if dist > proximity_thresh:
                continue
            if p1.team == p2.team and p1.team != "ball":
                edge_type = "teammate"
            elif p1.team != p2.team and "ball" not in (p1.team, p2.team):
                edge_type = "pressure"
            else:
                edge_type = "ball_proximity"
            G.add_edge(p1.track_id, p2.track_id, type=edge_type, distance=round(dist, 1))
            edges.append((p1.track_id, p2.track_id, edge_type))

    metrics.graph_edges = edges

    # ── Métricas de compactação ──────────────────────────────────────────
    def calc_team_metrics(pos_list):
        if len(pos_list) < 2:
            return (0, 0), 0, 0, 0
        arr = np.array(pos_list)
        centroid = arr.mean(axis=0)
        dists = np.linalg.norm(arr - centroid, axis=1)
        compactness = float(dists.mean())
        width = float(arr[:, 1].max() - arr[:, 1].min())   # eixo Y = largura
        depth = float(arr[:, 0].max() - arr[:, 0].min())   # eixo X = profundidade
        return tuple(centroid), compactness, width, depth

    metrics.team_a_centroid, metrics.team_a_compactness, \
        metrics.team_a_width, metrics.team_a_depth = calc_team_metrics(team_a_pos)
    metrics.team_b_centroid, metrics.team_b_compactness, \
        metrics.team_b_width, metrics.team_b_depth = calc_team_metrics(team_b_pos)

    # ── Zonas livres (grid simplificado) ─────────────────────────────────
    all_pos = team_a_pos + team_b_pos
    if all_pos:
        grid_x = np.linspace(0, 105, 8)
        grid_y = np.linspace(0, 68, 6)
        free_zones = []
        for gx in grid_x:
            for gy in grid_y:
                min_dist = min(np.hypot(gx - px, gy - py) for px, py in all_pos)
                if min_dist > 12.0:
                    free_zones.append((round(gx, 1), round(gy, 1)))
        metrics.free_zones = free_zones[:10]  # Top 10 zonas

    # ── Superioridade numérica por terço ─────────────────────────────────
    for zone_name, x_range in [("defesa", (0, 35)), ("meio", (35, 70)), ("ataque", (70, 105))]:
        a_count = sum(1 for x, y in team_a_pos if x_range[0] <= x < x_range[1])
        b_count = sum(1 for x, y in team_b_pos if x_range[0] <= x < x_range[1])
        if a_count > b_count:
            metrics.numerical_superiority[zone_name] = f"Team A ({a_count}x{b_count})"
        elif b_count > a_count:
            metrics.numerical_superiority[zone_name] = f"Team B ({b_count}x{a_count})"
        else:
            metrics.numerical_superiority[zone_name] = f"Equilibrado ({a_count}x{b_count})"

    # ── Linhas de passe bloqueadas ───────────────────────────────────────
    blocked = sum(1 for _, _, t in edges if t == "pressure")
    metrics.passing_lanes_blocked = blocked

    return G, metrics


# ── Processar todos os frames ────────────────────────────────────────────
all_graphs: Dict[float, nx.Graph] = {}
all_metrics: Dict[float, TacticalMetrics] = {}

for ts, positions in field_data.items():
    if len(positions) < 4:
        continue
    G, m = build_tactical_graph(positions)
    all_graphs[ts] = G
    all_metrics[ts] = m

print(f"🕸️  Grafos construídos: {len(all_graphs)} frames analisados")

# Preview de um frame
if all_metrics:
    sample_ts = sorted(all_metrics.keys())[len(all_metrics) // 2]
    sm = all_metrics[sample_ts]
    print(f"\n📊 Preview (t={sample_ts}s):")
    print(f"   Team A centroide: ({sm.team_a_centroid[0]:.0f}, {sm.team_a_centroid[1]:.0f}) | "
          f"Compactação: {sm.team_a_compactness:.1f}m")
    print(f"   Team B centroide: ({sm.team_b_centroid[0]:.0f}, {sm.team_b_centroid[1]:.0f}) | "
          f"Compactação: {sm.team_b_compactness:.1f}m")
    print(f"   Zonas livres: {len(sm.free_zones)} | Pressões: {sm.passing_lanes_blocked}")
    print(f"   Superioridade: {sm.numerical_superiority}")


In [ ]:
# =============================================================================
# CÉLULA 8 — LLM: ANÁLISE TÁTICA COM QWEN
# =============================================================================

from transformers import AutoModelForCausalLM, AutoTokenizer

def load_llm(model_name: str):
    """Carrega o modelo Qwen localmente."""
    print(f"🤖 Carregando LLM: {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True
    )
    print("✅ LLM carregado com sucesso.")
    return model, tokenizer


def prepare_tactical_prompt(metrics_dict: Dict[float, TacticalMetrics],
                            field_positions: Dict[float, List[FieldPosition]]) -> str:
    """
    Converte dados estruturados em prompt textual para o LLM.
    O LLM NUNCA recebe imagens — apenas dados numéricos e descritivos.
    """
    # Agregar métricas médias
    timestamps = sorted(metrics_dict.keys())
    if not timestamps:
        return "Dados insuficientes para análise."

    # Últimos frames (situação recente)
    recent = timestamps[-min(5, len(timestamps)):]
    recent_metrics = [metrics_dict[t] for t in recent]

    avg_a_compact = np.mean([m.team_a_compactness for m in recent_metrics])
    avg_b_compact = np.mean([m.team_b_compactness for m in recent_metrics])
    avg_a_width = np.mean([m.team_a_width for m in recent_metrics])
    avg_b_width = np.mean([m.team_b_width for m in recent_metrics])
    avg_a_depth = np.mean([m.team_a_depth for m in recent_metrics])
    avg_b_depth = np.mean([m.team_b_depth for m in recent_metrics])

    last_m = recent_metrics[-1]
    last_positions = field_positions.get(recent[-1], [])

    # Formatar posições
    team_a_positions_str = ""
    team_b_positions_str = ""
    for p in last_positions:
        if p.team == "team_a":
            team_a_positions_str += f"  - Player #{p.track_id}: ({p.x:.0f}, {p.y:.0f})\n"
        elif p.team == "team_b":
            team_b_positions_str += f"  - Player #{p.track_id}: ({p.x:.0f}, {p.y:.0f})\n"

    prompt = f"""You are an elite football tactical analyst working for a professional club's coaching staff.

Analyze the following structured data from a football match segment ({timestamps[0]}s to {timestamps[-1]}s).

## FIELD DIMENSIONS
- Field: 105m x 68m
- Team A (Brazil) attacks left-to-right
- Team B (Opponent) attacks right-to-left

## TEAM A (BRAZIL) — Current Positions:
{team_a_positions_str or '  No data available'}

## TEAM B (OPPONENT) — Current Positions:
{team_b_positions_str or '  No data available'}

## TACTICAL METRICS (averaged over last {len(recent)} frames):
- Team A compactness: {avg_a_compact:.1f}m (lower = more compact)
- Team B compactness: {avg_b_compact:.1f}m
- Team A width: {avg_a_width:.1f}m | depth: {avg_a_depth:.1f}m
- Team B width: {avg_b_width:.1f}m | depth: {avg_b_depth:.1f}m
- Team A centroid: ({last_m.team_a_centroid[0]:.0f}, {last_m.team_a_centroid[1]:.0f})
- Team B centroid: ({last_m.team_b_centroid[0]:.0f}, {last_m.team_b_centroid[1]:.0f})

## NUMERICAL SUPERIORITY BY ZONE:
- Defensive third: {last_m.numerical_superiority.get('defesa', 'N/A')}
- Midfield: {last_m.numerical_superiority.get('meio', 'N/A')}
- Attacking third: {last_m.numerical_superiority.get('ataque', 'N/A')}

## INTERACTION DATA:
- Pressure situations (opponent proximity): {last_m.passing_lanes_blocked}
- Free zones detected: {len(last_m.free_zones)} zones
- Free zone coordinates: {last_m.free_zones[:5]}

## REQUIRED OUTPUT:
Provide a professional tactical analysis in Portuguese (Brazilian) with:

1. **Análise da Formação**: Identify the likely formation of each team
2. **Problemas Identificados**: Key issues in Team A's structure
3. **Comportamento do Adversário**: How Team B is organizing defensively/offensively
4. **Espaços e Oportunidades**: Where are the exploitable spaces
5. **Recomendações Táticas**: Specific suggestions for Team A's coaching staff
6. **Ajustes para o Próximo Jogo**: Strategic adjustments

Be specific, reference player positions and zones. Write as if briefing a head coach."""

    return prompt


def run_llm_analysis(model, tokenizer, prompt: str, config: Config) -> str:
    """Executa o LLM e retorna a análise tática."""
    messages = [
        {"role": "system", "content": "You are a professional football tactical analyst."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=config.llm_max_tokens,
            temperature=config.llm_temperature,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1
        )

    response = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return response.strip()


# ── Executar ─────────────────────────────────────────────────────────────────
llm_model, llm_tokenizer = load_llm(cfg.llm_model_name)
tactical_prompt = prepare_tactical_prompt(all_metrics, field_data)
print(f"\n📝 Prompt gerado ({len(tactical_prompt)} chars). Enviando ao LLM...\n")

llm_analysis = run_llm_analysis(llm_model, llm_tokenizer, tactical_prompt, cfg)

print("=" * 80)
print("📋 ANÁLISE TÁTICA GERADA PELO LLM")
print("=" * 80)
print(llm_analysis)
print("=" * 80)

# Salvar análise
with open(os.path.join(cfg.output_dir, "tactical_analysis.txt"), "w", encoding="utf-8") as f:
    f.write(llm_analysis)
print(f"\n💾 Análise salva em {cfg.output_dir}/tactical_analysis.txt")


In [ ]:
from __future__ import annotations

# =============================================================================
# CÉLULA 9 — TACTICAL BOARD: VISUALIZAÇÃO PROFISSIONAL
# =============================================================================

from typing import List, Dict, Tuple, Optional
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch
import numpy as np
from dataclasses import dataclass, field
import textwrap
from scipy.spatial import ConvexHull
from scipy.stats import gaussian_kde

# Assuming FieldPosition, TacticalMetrics, and Config are defined elsewhere
# (e.g., in CÉLULA 2 and CÉLULA 7). For type hinting, they are re-declared here
# if the notebook context doesn't provide them in this cell's scope, but the
# prompt implies they are external and should not be redefined here.

# Re-declaring for type hinting within this cell's scope if they aren't globally available
# from previous cell executions for some reason, but typically they are.
# If you encounter NameError for these, ensure preceding cells are run.
@dataclass
class Config:
    video_path: str = ""
    fps_sample: int = 2
    max_duration: int = 39
    yolo_model: str = ""
    yolo_conf: float = 0.0
    yolo_person_class: int = 0
    yolo_ball_class: int = 0
    field_length: float = 105.0
    field_width: float = 68.0
    team_a_color: str = "#FFD700"
    team_b_color: str = "#1E90FF"
    ball_color: str = "#FF4444"
    llm_model_name: str = ""
    llm_max_tokens: int = 0
    llm_temperature: float = 0.0
    output_dir: str = "output"

@dataclass
class FieldPosition:
    x: float
    y: float
    team: str
    track_id: int
    timestamp: float

@dataclass
class TacticalMetrics:
    timestamp: float
    team_a_centroid: Tuple[float, float] = (0, 0)
    team_b_centroid: Tuple[float, float] = (0, 0)
    team_a_compactness: float = 0.0
    team_b_compactness: float = 0.0
    team_a_width: float = 0.0
    team_b_width: float = 0.0
    team_a_depth: float = 0.0
    team_b_depth: float = 0.0
    free_zones: List[Tuple[float, float]] = field(default_factory=list)
    numerical_superiority: Dict[str, str] = field(default_factory=dict)
    passing_lanes_blocked: int = 0
    graph_edges: List[Tuple[int, int, str]] = field(default_factory=list)

def draw_tactical_board(
    positions: List[FieldPosition],
    metrics: TacticalMetrics,
    analysis_text: str,
    config: Config,
    save_path: str = None,
) -> plt.Figure:
    """
    Renderiza um dashboard tático premium para análise de futebol.

    A visualização combina campo estilizado, pressão por KDE, convex hull,
    zonas livres, linhas de passe, shape das equipes, centroides, vetores
    curvos de movimentação, KPIs e insights da IA em um layout comercial.
    A função é defensiva: entradas vazias ou incompletas degradam sem erro.
    """
    positions = positions or []
    analysis_text = analysis_text or ""
    pitch_length = float(getattr(config, "field_length", 105.0) or 105.0)
    pitch_width = float(getattr(config, "field_width", 68.0) or 68.0)

    colors = {
        "bg": "#1A1A2E",
        "panel": "#161B33",
        "card": "#202642",
        "edge": "#33415C",
        "pitch_1": "#0B3D2E",
        "pitch_2": "#0F5138",
        "line": "#DDEBE4",
        "glow": "#2FE6A6",
        "team_a": getattr(config, "team_a_color", "#FFD700"),
        "team_b": getattr(config, "team_b_color", "#1E90FF"),
        "ball": getattr(config, "ball_color", "#FF4444"),
        "free": "#00F5A0",
        "good": "#00E676",
        "warn": "#FFD166",
        "bad": "#FF4D6D",
        "text": "#F8F9FA",
        "muted": "#B8C1D9",
        "dim": "#8D99AE",
    }

    def _safe_float(value: float, default: float = 0.0) -> float:
        """Retorna um float finito, usando fallback quando o dado é inválido."""
        try:
            result = float(value)
            return result if np.isfinite(result) else default
        except (ValueError, TypeError):
            return default

    def _clip_point(x_value: float, y_value: float) -> Tuple[float, float]:
        """Limita coordenadas aos limites do campo normalizado."""
        return (
            float(np.clip(_safe_float(x_value), 0.0, pitch_length)),
            float(np.clip(_safe_float(y_value), 0.0, pitch_width)),
        )

    def _team(team_name: str) -> List[FieldPosition]:
        """Filtra objetos FieldPosition por equipe."""
        return [p for p in positions if getattr(p, "team", None) == team_name]

    def _coords(players: List[FieldPosition]) -> np.ndarray:
        """Converte jogadores para array Nx2 de coordenadas seguras."""
        if not players:
            return np.empty((0, 2), dtype=float)
        return np.array([_clip_point(p.x, p.y) for p in players], dtype=float)

    def _meters(value: float) -> str:
        """Formata métricas em metros para cartões executivos."""
        value = _safe_float(value)
        return f"{value:.1f} m" if value > 0 else "—"

    def _status(value: float, good: float, warn: float) -> str:
        """Mapeia valor numérico para cor de status: excelente, atenção ou crítico."""
        value = _safe_float(value)
        if value <= 0:
            return colors["dim"]
        if value <= good:
            return colors["good"]
        if value <= warn:
            return colors["warn"]
        return colors["bad"]

    def _avg_distance(players: List[FieldPosition]) -> float:
        """Calcula a distância média par-a-par entre jogadores."""
        arr = _coords(players)
        if len(arr) < 2:
            return 0.0
        distances = [
            float(np.linalg.norm(arr[i] - arr[j]))
            for i in range(len(arr))
            for j in range(i + 1, len(arr))
        ]
        return float(np.mean(distances)) if distances else 0.0

    def _is_blocked(
        start: Tuple[float, float],
        end: Tuple[float, float],
        opponents: List[FieldPosition],
        threshold: float = 4.5,
    ) -> bool:
        """Detecta se adversários interceptam geometricamente uma linha de passe."""
        a = np.array(start, dtype=float)
        b = np.array(end, dtype=float)
        segment = b - a
        denom = float(np.dot(segment, segment))
        if denom <= 1e-9:
            return False
        for opponent in opponents:
            p = np.array(_clip_point(opponent.x, opponent.y), dtype=float)
            t = float(np.dot(p - a, segment) / denom)
            if 0.08 <= t <= 0.92:
                closest = a + t * segment
                if float(np.linalg.norm(p - closest)) <= threshold:
                    return True
        return False

    def _passing_lanes(
        players: List[FieldPosition],
        opponents: List[FieldPosition],
        max_distance: float = 27.0,
    ) -> List[Dict[str, object]]:
        """Gera linhas de passe próximas e classifica cada uma como livre ou bloqueada."""
        lanes = []
        for i, first in enumerate(players):
            for second in players[i + 1 :]:
                start = _clip_point(first.x, first.y)
                end = _clip_point(second.x, second.y)
                distance = float(np.hypot(start[0] - end[0], start[1] - end[1]))
                if 4.0 < distance <= max_distance:
                    lanes.append(
                        {
                            "start": start,
                            "end": end,
                            "distance": distance,
                            "blocked": _is_blocked(start, end, opponents),
                        }
                    )
        return sorted(lanes, key=lambda item: item["distance"])[:28]

    def _insights(text: str, limit: int = 5) -> List[str]:
        """Converte texto livre da LLM em bullets curtos e visualmente estáveis."""
        bullets = []
        for raw in str(text or "").replace("\r", "\n").split("\n"):
            line = raw.strip(" -*•\t")
            if not line or line.startswith("#"):
                continue
            if ":" in line and len(line.split(":", 1)[0]) <= 32:
                line = line.split(":", 1)[1].strip() or line
            bullets.extend(textwrap.wrap(line, width=52)[:2])
            if len(bullets) >= limit:
                break
        return bullets[:limit] or [
            "Dados insuficientes para inferência textual robusta.",
            "Validar espaços, pressão e shape no frame atual.",
        ]

    def _draw_pitch(ax: plt.Axes) -> None:
        """Desenha o gramado premium com faixas, brilho, linhas, áreas e gols."""
        ax.set_xlim(-5, pitch_length + 5)
        ax.set_ylim(-5, pitch_width + 5)
        ax.set_aspect("equal")
        ax.axis("off")
        ax.set_facecolor(colors["bg"])

        # Outer glow effect
        ax.add_patch(
            patches.FancyBboxPatch(
                (-1.8, -1.8), pitch_length + 3.6, pitch_width + 3.6,
                boxstyle="round,pad=0,rounding_size=2.2",
                fc="#000000", ec="none", alpha=0.30, zorder=0,
            )
        )

        # Grass stripes
        stripe_width = pitch_length / 14
        for idx in range(14):
            ax.add_patch(
                patches.Rectangle(
                    (idx * stripe_width, 0), stripe_width, pitch_width,
                    fc=colors["pitch_1"] if idx % 2 == 0 else colors["pitch_2"],
                    ec="none", zorder=1,
                )
            )

        # Subtle gradient overlay
        gradient = np.linspace(0.16, 0.0, 256).reshape(1, -1)
        ax.imshow(
            gradient, extent=(0, pitch_length, 0, pitch_width),
            cmap="Greens", alpha=0.13, origin="lower", zorder=2,
            aspect="auto",
        )

        # Inner glow for pitch lines
        ax.add_patch(
            patches.FancyBboxPatch(
                (0, 0), pitch_length, pitch_width,
                boxstyle="round,pad=0,rounding_size=1.5",
                fc="none", ec=colors["glow"], lw=5, alpha=0.10, zorder=3,
            )
        )

        line_color = colors["line"]

        # Outer lines
        ax.plot(
            [0, pitch_length, pitch_length, 0, 0], [0, 0, pitch_width, pitch_width, 0],
            color=line_color, lw=1.6, alpha=0.92, zorder=6,
        )

        # Halfway line
        ax.plot(
            [pitch_length / 2, pitch_length / 2], [0, pitch_width],
            color=line_color, lw=1.2, alpha=0.82, zorder=6,
        )

        # Center circle and spot
        ax.add_patch(
            patches.Circle((pitch_length / 2, pitch_width / 2), 9.15,
                            fill=False, ec=line_color, lw=1.2, alpha=0.82, zorder=6)
        )
        ax.add_patch(
            patches.Circle((pitch_length / 2, pitch_width / 2), 0.55,
                            fc=line_color, ec="none", alpha=0.90, zorder=7)
        )

        # Penalty areas, goal areas, and penalty spots
        penalty_y = (pitch_width - 40.32) / 2
        goal_area_y = (pitch_width - 18.32) / 2
        for side in (0.0, pitch_length):
            direction = 1 if side == 0 else -1
            penalty_x = side if side == 0 else side - 16.5
            goal_x = side if side == 0 else side - 5.5

            # Penalty area
            ax.add_patch(
                patches.Rectangle((penalty_x, penalty_y), 16.5, 40.32,
                                   fill=False, ec=line_color, lw=1.2, alpha=0.82, zorder=6)
            )
            # Goal area
            ax.add_patch(
                patches.Rectangle((goal_x, goal_area_y), 5.5, 18.32,
                                   fill=False, ec=line_color, lw=1.2, alpha=0.82, zorder=6)
            )
            # Penalty spot
            spot = (side + direction * 11.0, pitch_width / 2)
            ax.add_patch(
                patches.Circle(spot, 0.45, fc=line_color, ec="none", alpha=0.90, zorder=7)
            )
            # Penalty arc
            theta1, theta2 = (-53, 53) if side == 0 else (127, 233)
            ax.add_patch(
                patches.Arc(spot, 18.3, 18.3, theta1=theta1, theta2=theta2,
                             ec=line_color, lw=1.2, alpha=0.70, zorder=6)
            )

        # Goals
        goal_y = (pitch_width - 7.32) / 2
        ax.add_patch(
            patches.Rectangle((-2.2, goal_y), 2.2, 7.32,
                               fc="#0A0D18", ec=line_color, lw=1.0, alpha=0.85, zorder=5)
        )
        ax.add_patch(
            patches.Rectangle((pitch_length, goal_y), 2.2, 7.32,
                               fc="#0A0D18", ec=line_color, lw=1.0, alpha=0.85, zorder=5)
        )

        # Thirds lines (subtle)
        for x_mark in (pitch_length / 3, 2 * pitch_length / 3):
            ax.plot(
                [x_mark, x_mark], [0, pitch_width], color="#FFFFFF",
                lw=0.7, alpha=0.12, ls="--", zorder=5,
            )

    def _draw_pressure_map(ax: plt.Axes) -> None:
        """Renderiza mapa de pressão translúcido via Kernel Density Estimation."""
        players = [p for p in positions if getattr(p, "team", None) in {"team_a", "team_b"}]
        arr = _coords(players)
        if len(arr) < 3:
            return
        try:
            x_grid, y_grid = np.mgrid[0:pitch_length:180j, 0:pitch_width:120j]
            kde = gaussian_kde(np.vstack([arr[:, 0], arr[:, 1]]))
            density = kde(np.vstack([x_grid.ravel(), y_grid.ravel()])).reshape(x_grid.shape)
            if float(density.max()) > 0:
                density = density / density.max()
            ax.contourf(x_grid, y_grid, density, levels=np.linspace(0.35, 1.0, 9),
                        cmap="inferno", alpha=0.23, antialiased=True, zorder=4)
        except Exception:
            return  # Fail gracefully if KDE calculation fails

    def _draw_convex_hulls(
        ax: plt.Axes,
        team_a: List[FieldPosition],
        team_b: List[FieldPosition],
    ) -> None:
        """Desenha hulls convexos elegantes das equipes, tratando falhas de geometria."""
        for players, color, label in (
            (team_a, colors["team_a"], "TEAM A SHAPE"),
            (team_b, colors["team_b"], "TEAM B SHAPE"),
        ):
            arr = _coords(players)
            if len(arr) < 3:
                continue
            try:
                hull_points = arr[ConvexHull(arr).vertices]
                ax.add_patch(
                    patches.Polygon(
                        hull_points, closed=True, fc=color, ec=color,
                        lw=1.2, alpha=0.13, zorder=8,
                    )
                )
                ax.add_patch(
                    patches.Polygon(
                        hull_points, closed=True, fill=False, ec=color,
                        lw=1.35, alpha=0.62, zorder=9,
                    )
                )
                # Labeling the convex hull (optional, can be removed if too cluttered)
                top_point_idx = int(np.argmax(hull_points[:, 1]))
                top = hull_points[top_point_idx]
                ax.text(top[0], top[1] + 2.0, label, color=color, fontsize=7.5,
                        fontweight="bold", ha="center", va="bottom", alpha=0.86, zorder=10)
            except Exception:
                continue  # Fail gracefully if ConvexHull fails

    def _draw_free_zones(ax: plt.Axes) -> None:
        """Destaca espaços livres com círculos translúcidos, contorno verde e rótulo."""
        for zone in list(getattr(metrics, "free_zones", []) or [])[:10]: # Limit to top 10 zones
            try:
                x_zone, y_zone = _clip_point(zone[0], zone[1])
            except Exception:
                continue
            ax.add_patch(
                patches.Circle((x_zone, y_zone), 7.0, fc=colors["free"],
                               ec="none", alpha=0.07, zorder=10)
            )
            ax.add_patch(
                patches.Circle((x_zone, y_zone), 4.8, fc=colors["free"],
                               ec=colors["free"], lw=1.1, alpha=0.18, zorder=11)
            )
            ax.add_patch(
                patches.Circle((x_zone, y_zone), 4.8, fill=False,
                               ec="#D8FFF0", lw=0.8, ls="--", alpha=0.75, zorder=12)
            )
            ax.text(x_zone, y_zone, "FREE\nSPACE", color="#D8FFF0", fontsize=6.4,
                    fontweight="bold", ha="center", va="center", linespacing=0.85, zorder=13)

    def _draw_passing_lanes(
        ax: plt.Axes,
        team_a: List[FieldPosition],
        team_b: List[FieldPosition],
    ) -> Tuple[int, int]:
        """Desenha linhas de passe livres em verde e bloqueadas em vermelho."""
        free_count, blocked_count = 0, 0
        for lane in _passing_lanes(team_a, team_b):
            blocked = bool(lane["blocked"])
            blocked_count += int(blocked)
            free_count += int(not blocked)
            start, end = lane["start"], lane["end"]
            ax.plot(
                [start[0], end[0]], [start[1], end[1]],
                color=colors["bad"] if blocked else colors["good"],
                lw=1.15, alpha=0.42 if blocked else 0.34,
                ls="--" if blocked else "-", zorder=14,
            )
        return free_count, blocked_count

    def _draw_defensive_lines(
        ax: plt.Axes,
        players: List[FieldPosition],
        color: str,
    ) -> None:
        """Agrupa jogadores por profundidade e desenha linhas do bloco defensivo."""
        arr = _coords(players)
        if len(arr) < 3:
            return
        try:
            # Define three approximate thirds of the pitch for depth grouping
            qs = np.quantile(arr[:, 0], [0.25, 0.75]) # Quartiles for better grouping
            groups = [
                arr[arr[:, 0] <= qs[0]],
                arr[(arr[:, 0] > qs[0]) & (arr[:, 0] <= qs[1])],
                arr[arr[:, 0] > qs[1]],
            ]

            for group in groups:
                if len(group) < 2:
                    continue
                x_line = float(np.mean(group[:, 0]))
                y_min, y_max = float(np.min(group[:, 1])), float(np.max(group[:, 1]))

                ax.plot(
                    [x_line, x_line], [y_min, y_max],
                    color=color, lw=2.0, alpha=0.20, zorder=10,
                )
                ax.plot(
                    [x_line - 1.2, x_line + 1.2], [y_min, y_min],
                    color=color, lw=1.0, alpha=0.25, zorder=10,
                )
                ax.plot(
                    [x_line - 1.2, x_line + 1.2], [y_max, y_max],
                    color=color, lw=1.0, alpha=0.25, zorder=10,
                )
        except Exception:
            return  # Fail gracefully if calculation fails

    def _draw_team_shape(
        ax: plt.Axes,
        players: List[FieldPosition],
        color: str,
        label: str,
    ) -> None:
        """Exibe largura, profundidade e caixa de shape da equipe."""
        arr = _coords(players)
        if len(arr) < 2:
            return
        x_min, y_min = arr.min(axis=0)
        x_max, y_max = arr.max(axis=0)
        width, depth = y_max - y_min, x_max - x_min

        ax.add_patch(
            patches.Rectangle((x_min, y_min), max(depth, 0.1), max(width, 0.1),
                               fill=False, ec=color, lw=0.9, ls=":", alpha=0.45, zorder=11)
        )
        ax.text(
            (x_min + x_max) / 2, min(pitch_width + 2.7, y_max + 1.4), # Adjust y for visibility
            f"{label}  W {width:.1f}m | D {depth:.1f}m",
            color=color, fontsize=7.2, fontweight="bold", ha="center", va="bottom",
            alpha=0.80, zorder=12,
        )

    def _draw_team_centroids(
        ax: plt.Axes,
        team_a: List[FieldPosition],
        team_b: List[FieldPosition],
    ) -> None:
        """Marca centroides e distância espacial entre blocos das equipes."""
        specs = []
        if team_a:
            specs.append((_clip_point(*getattr(metrics, "team_a_centroid", (0,0))), colors["team_a"], "A-CENTROID"))
        if team_b:
            specs.append((_clip_point(*getattr(metrics, "team_b_centroid", (0,0))), colors["team_b"], "B-CENTROID"))

        if len(specs) == 2:
            a, b = specs[0][0], specs[1][0]
            ax.plot([a[0], b[0]], [a[1], b[1]], color="#FFFFFF", lw=1.0, alpha=0.18, ls="--", zorder=13)

        for (x_centroid, y_centroid), color, label in specs:
            ax.add_patch(
                patches.Circle((x_centroid, y_centroid), 4.0, fc=color, ec="none", alpha=0.10, zorder=23)
            )
            ax.add_patch(
                patches.Circle((x_centroid, y_centroid), 2.3, fc=color, ec="white", lw=1.25, alpha=0.95, zorder=24)
            )
            ax.text(x_centroid, y_centroid + 4.5, label, color="white", fontsize=6.8,
                    fontweight="bold", ha="center", va="bottom", zorder=25)

    def _draw_movement_vectors(ax: plt.Axes, team_a: List[FieldPosition]) -> None:
        """Desenha vetores curvos de movimentação com FancyArrowPatch."""
        zones = list(getattr(metrics, "free_zones", []) or [])
        if not team_a or not zones:
            return

        start = _clip_point(*getattr(metrics, "team_a_centroid", (0,0))) # From Team A centroid
        for idx, zone in enumerate(zones[:3]): # Draw vectors to top 3 free zones
            try:
                end = _clip_point(zone[0], zone[1])
            except Exception:
                continue
            ax.add_patch(
                FancyArrowPatch(
                    start, end, arrowstyle="-|>", mutation_scale=15,
                    connectionstyle=f"arc3,rad={0.12 + idx * 0.05}", lw=2.0,
                    linestyle="--", color=colors["team_a"], alpha=0.62, zorder=21,
                )
            )

    def _draw_players(ax: plt.Axes) -> None:
        """Desenha jogadores com sombra, glow, borda branca e número centralizado."""
        for player in positions:
            team = getattr(player, "team", "unknown")
            x_player, y_player = _clip_point(getattr(player, "x", 0), getattr(player, "y", 0))

            # Determine marker, color, and size based on team
            if team == "team_a":
                marker, color, size = "o", colors["team_a"], 245
            elif team == "team_b":
                marker, color, size = "s", colors["team_b"], 225
            elif team == "ball":
                marker, color, size = "*", colors["ball"], 360
            else: # Unknown or untracked players
                marker, color, size = "^", colors["dim"], 155

            # Shadow effect
            ax.scatter(x_player + 0.7, y_player - 0.7, s=size * 1.15, marker=marker,
                       c="#000000", alpha=0.28, linewidths=0, zorder=26)
            # Glow effect
            ax.scatter(x_player, y_player, s=size * 1.55, marker=marker,
                       c=color, alpha=0.14, linewidths=0, zorder=27)
            # Main player marker with white border
            ax.scatter(x_player, y_player, s=size, marker=marker, c=color,
                       edgecolors="white", linewidths=1.35, alpha=0.98, zorder=28)

            # Player number or ball indicator
            label = "●" if team == "ball" else str(getattr(player, "track_id", ""))[-2:] # Last two digits of track_id
            ax.text(x_player, y_player, label,
                    color="#10131F" if team in {"team_a", "ball"} else "white", # Dark text for yellow/red, white for blue
                    fontsize=7.4 if team != "ball" else 8.5, fontweight="bold",
                    ha="center", va="center", zorder=29)

    def _draw_legend(ax: plt.Axes) -> None:
        """Adiciona legenda elegante no rodapé interno do campo."""
        # Define legend items (color, marker, label)
        items = [
            (colors["team_a"], "o", "Team A"),
            (colors["team_b"], "s", "Team B"),
            (colors["ball"], "*", "Ball"),
            (colors["good"], "_", "Open lane"),
            (colors["bad"], "_", "Blocked lane"),
        ]

        # Position the legend box
        x_start, y_start = 2.0, -3.2
        ax.add_patch(
            patches.FancyBboxPatch((x_start - 1.2, y_start - 1.25), 58.5, 3.0,
                                   boxstyle="round,pad=0.35,rounding_size=0.8",
                                   fc="#101525", ec=colors["edge"], lw=0.9,
                                   alpha=0.86, zorder=30)
        )

        # Add each item to the legend
        cursor = x_start
        for color, marker, label in items:
            if marker == "_":  # Line for passing lanes
                ax.plot([cursor, cursor + 2.3], [y_start, y_start], color=color, lw=2.4, alpha=0.85, zorder=31)
                text_x, cursor = cursor + 3.1, cursor + 14.2
            else: # Marker for players/ball
                ax.scatter(cursor, y_start, s=72, marker=marker, c=color, edgecolors="white", linewidths=0.7, zorder=31)
                text_x, cursor = cursor + 1.8, cursor + 11.3
            ax.text(text_x, y_start, label, color=colors["muted"], fontsize=7.4,
                    fontweight="bold", va="center", zorder=31)

    def _draw_card(
        ax: plt.Axes, x: float, y: float, w: float, h: float,
        title: str, value: str, icon: str, accent: str,
    ) -> None:
        """Desenha um cartão KPI com ícone Unicode e faixa de status."""
        # Card background
        ax.add_patch(
            patches.FancyBboxPatch((x, y), w, h,
                                   boxstyle="round,pad=0.014,rounding_size=0.028",
                                   fc=colors["card"], ec=colors["edge"], lw=1.0,
                                   alpha=0.96, transform=ax.transAxes, zorder=2)
        )
        # Accent strip on the left
        ax.add_patch(
            patches.FancyBboxPatch((x, y), 0.018, h,
                                   boxstyle="round,pad=0,rounding_size=0.018",
                                   fc=accent, ec="none", alpha=0.95,
                                   transform=ax.transAxes, zorder=3)
        )
        # Icon
        ax.text(x + 0.045, y + h - 0.035, icon, color=accent, fontsize=13,
                fontweight="bold", transform=ax.transAxes, va="top", zorder=4)
        # Title
        ax.text(x + 0.105, y + h - 0.032, title, color=colors["muted"], fontsize=7.6,
                fontweight="bold", transform=ax.transAxes, va="top", zorder=4)
        # Value
        ax.text(x + 0.045, y + 0.030, value, color=colors["text"], fontsize=13.5,
                fontweight="bold", transform=ax.transAxes, va="bottom", zorder=4)
        # Separator line
        ax.plot([x + 0.045, x + w - 0.035], [y + 0.026, y + 0.026],
                color=accent, lw=1.0, alpha=0.35, transform=ax.transAxes, zorder=4)

    def _draw_dashboard(ax: plt.Axes, passing_stats: Tuple[int, int]) -> None:
        """Renderiza painel lateral com cartões, superioridade numérica e insights."""
        ax.set_facecolor(colors["panel"])
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis("off")

        # Panel border
        ax.add_patch(
            patches.FancyBboxPatch((0, 0), 1, 1,
                                   boxstyle="round,pad=0.012,rounding_size=0.035",
                                   fc=colors["panel"], ec=colors["edge"], lw=1.4,
                                   transform=ax.transAxes, zorder=1)
        )

        # Title and subtitle
        ax.text(0.055, 0.965, "TACTICAL COMMAND CENTER", color=colors["text"],
                fontsize=12.5, fontweight="bold", transform=ax.transAxes, va="top", zorder=5)
        ax.text(0.055, 0.928, "Live spatial intelligence dashboard", color=colors["dim"],
                fontsize=7.8, transform=ax.transAxes, va="top", zorder=5)

        # --- Prepare KPI data ---
        compact_a = _safe_float(getattr(metrics, "team_a_compactness", 0.0))
        width_a = _safe_float(getattr(metrics, "team_a_width", 0.0))
        depth_a = _safe_float(getattr(metrics, "team_a_depth", 0.0))
        free_count = len(getattr(metrics, "free_zones", []) or [])
        blocked_lanes = int(passing_stats[1]) if passing_stats else int(getattr(metrics, "passing_lanes_blocked", 0) or 0)
        superiority_data = getattr(metrics, "numerical_superiority", {}) or {}
        superiority_count = sum("Team A" in str(v) for v in superiority_data.values())
        # Simple heuristic for pressure
        pressure_status = "Alta" if blocked_lanes >= 8 else "Média" if blocked_lanes >= 4 else "Baixa"
        avg_dist_a = _avg_distance(_team("team_a"))

        # KPI Cards data
        cards = [
            ("COMPACTAÇÃO", _meters(compact_a), "◈", _status(compact_a, 18, 28)),
            ("LARGURA", _meters(width_a), "↔", colors["team_a"]),
            ("PROFUNDIDADE", _meters(depth_a), "↕", colors["team_b"]),
            ("PRESSÃO", pressure_status, "▲", colors["bad"] if pressure_status == "Alta" else colors["warn"]),
            ("ZONAS LIVRES", str(free_count), "◎", colors["free"]),
            ("PASSES BLOQ.", str(blocked_lanes), "×", colors["bad"] if blocked_lanes else colors["good"]),
            ("SUPERIORIDADE", f"{superiority_count}/3", "⚔", colors["good"] if superiority_count > 0 else colors["dim"]),
            ("DIST. MÉDIA", _meters(avg_dist_a), "∑", colors["warn"] if avg_dist_a > 15 else colors["good"]),
        ]

        # Draw KPI cards (2 columns)
        for idx, (title, value, icon, accent) in enumerate(cards):
            _draw_card(ax, [0.055, 0.525][idx % 2], 0.835 - (idx // 2) * 0.105,
                       0.425, 0.088, title, value, icon, accent)

        # Numerical Superiority section
        sup_y = 0.365
        ax.text(0.055, sup_y, "NUMERICAL SUPERIORITY", color=colors["text"],
                fontsize=9.5, fontweight="bold", transform=ax.transAxes, va="top", zorder=5)
        ax.plot([0.055, 0.945], [sup_y - 0.020, sup_y - 0.020], color=colors["edge"],
                lw=1.0, transform=ax.transAxes, zorder=5)

        for idx, (key, short) in enumerate((("defesa", "DEF"), ("meio", "MID"), ("ataque", "ATT"))):
            value_str = str(superiority_data.get(key, "N/A"))
            accent_color = colors["good"] if "Team A" in value_str else colors["bad"] if "Team B" in value_str else colors["warn"]
            y_pos = sup_y - 0.055 - idx * 0.045
            ax.text(0.060, y_pos, short, color=accent_color, fontsize=8.4, fontweight="bold",
                    transform=ax.transAxes, va="center", zorder=5)
            ax.text(0.170, y_pos, value_str[:40], color=colors["muted"], fontsize=8.0,
                    transform=ax.transAxes, va="center", zorder=5)

        # Tactical Insights from LLM
        ax.add_patch(
            patches.FancyBboxPatch((0.055, 0.040), 0.890, 0.145,
                                   boxstyle="round,pad=0.018,rounding_size=0.026",
                                   fc="#101525", ec=colors["edge"], lw=1.0,
                                   transform=ax.transAxes, zorder=2)
        )
        ax.text(0.080, 0.195, "TACTICAL INSIGHTS", color=colors["text"], fontsize=9.5,
                fontweight="bold", transform=ax.transAxes, va="top", zorder=5)
        for idx, insight in enumerate(_insights(analysis_text, limit=5)):
            ax.text(0.083, 0.165 - idx * 0.0215, f"• {insight}", color=colors["muted"],
                    fontsize=7.4, transform=ax.transAxes, va="top", zorder=5)

    def _draw_footer(fig_obj: plt.Figure, ax: plt.Axes) -> None:
        """Adiciona timeline inferior com frame, timestamp e branding discreto."""
        ax.set_facecolor(colors["bg"])
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis("off")

        # Branding text
        ax.text(0.025, 0.68, "Football Tactical AI", color=colors["text"], fontsize=12.5,
                fontweight="bold", transform=ax.transAxes, va="center")
        ax.text(0.025, 0.32, "Computer Vision + Graph AI + LLM  ·  Proof of Concept",
                color=colors["dim"], fontsize=8.5, transform=ax.transAxes, va="center")

        # Timestamp and Frame
        timestamp = _safe_float(getattr(metrics, "timestamp", 0.0))
        fps = _safe_float(getattr(config, "fps_sample", 2), 2.0) # Ensure fps is float and not zero
        frame = int(round(timestamp * max(1.0, fps)))

        ax.text(0.525, 0.68, f"FRAME {frame:04d}", color=colors["muted"], fontsize=8.8,
                fontweight="bold", transform=ax.transAxes, va="center")
        ax.text(0.675, 0.68, f"TIMESTAMP {timestamp:05.2f}s", color=colors["muted"],
                fontsize=8.8, fontweight="bold", transform=ax.transAxes, va="center")

        # Progress bar (timeline)
        max_duration = _safe_float(getattr(config, "max_duration", 39), 39.0)
        progress = float(np.clip(timestamp / max(1.0, max_duration), 0.0, 1.0))
        x0, x1, y = 0.525, 0.965, 0.32
        ax.plot([x0, x1], [y, y], color=colors["edge"], lw=5, alpha=0.85,
                solid_capstyle="round", transform=ax.transAxes)
        ax.plot([x0, x0 + (x1 - x0) * progress], [y, y], color=colors["glow"], lw=5,
                alpha=0.92, solid_capstyle="round", transform=ax.transAxes)
        ax.scatter([x0 + (x1 - x0) * progress], [y], s=52, c=colors["glow"],
                   edgecolors="white", linewidths=0.8, transform=ax.transAxes, zorder=5)

        # Top right corner label
        fig_obj.text(0.985, 0.985, "PRO ANALYTICS VIEW", color="#FFFFFF", alpha=0.42,
                     fontsize=7.5, ha="right", va="top", fontweight="bold")

    # --- Main Figure Setup ---
    team_a_players = _team("team_a")
    team_b_players = _team("team_b")

    fig = plt.figure(figsize=(22, 14), facecolor=colors["bg"])

    # Define subplots axes
    ax_field = fig.add_axes([0.025, 0.105, 0.660, 0.810])    # Main field plot
    ax_dashboard = fig.add_axes([0.705, 0.145, 0.270, 0.755]) # Right dashboard panel
    ax_footer = fig.add_axes([0.025, 0.025, 0.950, 0.055])   # Bottom footer/timeline

    # Global titles
    fig.text(0.025, 0.965, "TACTICAL BOARD", color=colors["text"], fontsize=24,
             fontweight="bold", ha="left", va="top")
    fig.text(0.025, 0.935, "Professional football spatial analysis dashboard",
             color=colors["dim"], fontsize=10.5, ha="left", va="top")
    fig.text(0.675, 0.965, "HUDL / WYSCOUT / STATSBOMB STYLE POC",
             color=colors["glow"], fontsize=8.8, fontweight="bold", ha="right",
             va="top", alpha=0.80)

    # --- Drawing elements on the field axis ---
    _draw_pitch(ax_field)
    _draw_pressure_map(ax_field)
    _draw_convex_hulls(ax_field, team_a_players, team_b_players)
    _draw_free_zones(ax_field)
    passing_stats = _draw_passing_lanes(ax_field, team_a_players, team_b_players)
    _draw_defensive_lines(ax_field, team_a_players, colors["team_a"])
    _draw_defensive_lines(ax_field, team_b_players, colors["team_b"])
    _draw_team_shape(ax_field, team_a_players, colors["team_a"], "TEAM A")
    _draw_team_shape(ax_field, team_b_players, colors["team_b"], "TEAM B")
    _draw_movement_vectors(ax_field, team_a_players)
    _draw_team_centroids(ax_field, team_a_players, team_b_players)
    _draw_players(ax_field)
    _draw_legend(ax_field)

    # --- Drawing elements on the dashboard axis ---
    _draw_dashboard(ax_dashboard, passing_stats)

    # --- Drawing elements on the footer axis ---
    _draw_footer(fig, ax_footer)

    # --- Save and Return ---
    if save_path:
        try:
            fig.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.25,
                        facecolor=fig.get_facecolor(), edgecolor="none")
            print(f"🖼️  Imagem salva: {save_path}")
        except Exception as exc:
            print(f"⚠️  Não foi possível salvar a imagem em {save_path}: {exc}")

    return fig

In [ ]:
import matplotlib.pyplot as plt

# Get the last timestamp and its corresponding metrics and positions
last_timestamp = sorted(all_metrics.keys())[-1]
last_metrics = all_metrics[last_timestamp]
last_positions = field_data[last_timestamp]

# Generate the tactical board
fig = draw_tactical_board(
    positions=last_positions,
    metrics=last_metrics,
    analysis_text=llm_analysis,
    config=cfg,
    save_path=os.path.join(cfg.output_dir, "tactical_board.png")
)

from IPython.display import Image, display
display(Image(filename=os.path.join(cfg.output_dir, "tactical_board.png")))

# Display the generated figure
plt.show()

In [ ]:
# =============================================================================
# CÉLULA 10 — RELATÓRIO FINAL CONSOLIDADO
# =============================================================================

def generate_final_report(analysis: str,
                          metrics_dict: Dict[float, TacticalMetrics],
                          config: Config) -> str:
    """Gera o relatório final consolidado em Markdown."""

    timestamps = sorted(metrics_dict.keys())
    if not timestamps:
        return "Dados insuficientes."

    # Agregar métricas
    all_m = list(metrics_dict.values())
    avg_a_comp = np.mean([m.team_a_compactness for m in all_m])
    avg_b_comp = np.mean([m.team_b_compactness for m in all_m])
    avg_pressures = np.mean([m.passing_lanes_blocked for m in all_m])
    total_free_zones = np.mean([len(m.free_zones) for m in all_m])

    report = f"""# ⚽ FOOTBALL TACTICAL AI — RELATÓRIO FINAL
## Proof of Concept | Análise Automatizada por IA

---

### 📌 Parâmetros da Análise
- **Duração analisada:** {timestamps[0]}s → {timestamps[-1]}s
- **Frames processados:** {len(timestamps)}
- **Modelo de detecção:** {config.yolo_model}
- **LLM utilizado:** {config.llm_model_name}
- **Campo normalizado:** {config.field_length}m × {config.field_width}m

---

### 📊 Métricas Agregadas (médias)

| Métrica | Team A (Brasil) | Team B (Adversário) |
|---------|:----------------:|:-------------------:|
| Compactação | {avg_a_comp:.1f}m | {avg_b_comp:.1f}m |
| Pressões ativas | — | {avg_pressures:.1f} por frame |
| Zonas livres | {total_free_zones:.1f} por frame | — |

---

### 🤖 Análise Tática (gerada pelo LLM)

{analysis}

---

### 🔧 Arquitetura do Sistema

"""
    return report

# Exemplo de uso
report_final = generate_final_report(llm_analysis, all_metrics, cfg)
print(report_final)

with open(os.path.join(cfg.output_dir, "final_report.md"), "w", encoding="utf-8") as f:
    f.write(report_final)
print(f"\n💾 Relatório final salvo em {cfg.output_dir}/final_report.md")

Vídeo → OpenCV → YOLO (detecção) → Tracking → Field Mapping → Graph Builder → LLM (Qwen) → Insight Generator → Tactical Board